In [1]:
# =========================
# 📦 1. IMPORTS
# =========================
import pandas as pd
import xmlrpc.client
import re

In [10]:
# =========================
# 🔌 2. CONEXÃO ODOO
# =========================
url = "https://crm.brainess.com.br/"
db = "sohome_18"
username = "tiago@sohome.com"
password = "admin"

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")
print("✅ Conectado ao Odoo")

✅ Conectado ao Odoo


In [15]:
# =========================
# 📄 3. CARREGAR PLANILHA
# =========================
df = pd.read_excel(r"C:\Users\tiago.premiano\Downloads\CADASTRO_CLIENTES.xlsx")

df.columns = [
    "cliente",
    "representante",
    "cnpj",
    "coordenador",
    "email_coordenador",
    "email_representante"
    
]

In [18]:
df = df.loc[df["cliente"].notna() & df["email_coordenador"].notna()].copy()

In [20]:
print(f"📊 Total de linhas: {len(df)}")


# =========================
# 🧠 4. CACHE (performance)
# =========================
user_cache = {}
partner_cache = {}


# =========================
# 🔧 5. FUNÇÕES AUXILIARES
# =========================

def clean_cnpj(cnpj):
    return re.sub(r'\D', '', str(cnpj))


def get_or_create_user(nome, email):
    email = str(email).strip().lower()

    if not email or email == "nan":
        raise Exception("Email inválido")

    # Cache
    if email in user_cache:
        return user_cache[email]

    # Buscar no Odoo
    users = models.execute_kw(
        db, uid, password,
        'res.users', 'search',
        [[['login', '=', email]]]
    )

    if users:
        user_id = users[0]

        # Atualizar dados
        models.execute_kw(
            db, uid, password,
            'res.users', 'write',
            [[user_id], {
                'name': nome,
                'email': email
            }]
        )

        user_cache[email] = user_id
        return user_id

    # Criar usuário
    user_id = models.execute_kw(
        db, uid, password,
        'res.users', 'create',
        [{
            'name': nome,
            'login': email,
            'email': email,
        }]
    )

    user_cache[email] = user_id
    return user_id


def set_manager(user_id, manager_id):
    current = models.execute_kw(
        db, uid, password,
        'res.users', 'read',
        [[user_id], ['parent_id']]
    )[0]

    current_manager = current['parent_id'][0] if current['parent_id'] else None

    if current_manager != manager_id:
        models.execute_kw(
            db, uid, password,
            'res.users', 'write',
            [[user_id], {'parent_id': manager_id}]
        )


def get_or_create_partner(nome, cnpj, user_id):
    cnpj = clean_cnpj(cnpj)

    if not cnpj:
        raise Exception("CNPJ vazio")

    # Cache
    if cnpj in partner_cache:
        return partner_cache[cnpj]

    partners = models.execute_kw(
        db, uid, password,
        'res.partner', 'search',
        [[['vat', '=', cnpj]]]
    )

    if partners:
        partner_id = partners[0]

        # Atualizar vendedor
        models.execute_kw(
            db, uid, password,
            'res.partner', 'write',
            [[partner_id], {'user_id': user_id}]
        )

        partner_cache[cnpj] = partner_id
        return partner_id

    # Criar cliente
    partner_id = models.execute_kw(
        db, uid, password,
        'res.partner', 'create',
        [{
            'name': nome,
            'vat': cnpj,
            'user_id': user_id
        }]
    )

    partner_cache[cnpj] = partner_id
    return partner_id


# =========================
# 🚀 6. PROCESSAMENTO
# =========================
sucesso = 0
erro = 0

for i, row in df.iterrows():
    try:
        nome_cliente = str(row["cliente"]).strip()
        nome_rep = str(row["representante"]).strip()
        nome_coord = str(row["coordenador"]).strip()
        email_rep = str(row["email_representante"]).strip()
        email_coord = str(row["email_coordenador"]).strip()
        cnpj = row["cnpj"]

        # Validação mínima
        if not email_rep or not email_coord:
            raise Exception("Email faltando")

        # Coordenador
        coord_id = get_or_create_user(nome_coord, email_coord)

        # Representante
        rep_id = get_or_create_user(nome_rep, email_rep)

        # Hierarquia
        set_manager(rep_id, coord_id)

        # Cliente
        partner_id = get_or_create_partner(nome_cliente, cnpj, rep_id)

        sucesso += 1
        print(f"✅ [{i}] {nome_cliente}")

    except Exception as e:
        erro += 1
        print(f"❌ [{i}] ERRO: {row.get('cliente')} -> {e}")


# =========================
# 📊 7. RESULTADO FINAL
# =========================
print("\n=========================")
print(f"✅ Sucesso: {sucesso}")
print(f"❌ Erros: {erro}")
print("=========================")

📊 Total de linhas: 6909
❌ [1] ERRO: A ALTO ASTRAL CORTINAS E DECORACOES LTDA -> Request-sent
✅ [2] A BRASILEIRA
✅ [3] ACCERVO DESIGN
✅ [4] ACCERVO DESIGN
✅ [5] ADORI CASA
✅ [6] ADRIANA OLIVEIRA MENDES - FESTAS
✅ [7] AFECTTO MOVEIS LTDA
✅ [8] AGAPE HOME COMERCIO DE MOVEIS E DECORACOES LTDA
✅ [9] ALUMINAS MOVEIS LTDA
✅ [10] AMAIS
✅ [11] AMAIS
✅ [12] AMAIS
✅ [13] AMAIS
✅ [14] AMAIS
✅ [15] AMAIS
✅ [16] AMBIENTES
✅ [17] AMBIENTES MOVEIS E DECORACOES LTDA
✅ [18] AMBIENTI DECOR MOVEIS LTDA
✅ [19] AMEM CASA
✅ [20] ANDRADE MOVEIS CONCEITO
✅ [21] ANDRADE MOVEIS CONCEITO
✅ [22] ANDREIA ROCHA COMERCIO DE DECORACOES LTDA.
✅ [23] APOLO CASA
✅ [24] APOLO CASA
✅ [25] ARCADIA ART & DECOR
✅ [26] ARMAZEM DO MOVEL
✅ [27] ARQUE MOVEIS
✅ [28] ARQUE MOVEIS
✅ [29] ARRANJO MOVEIS
✅ [30] ARTE DO SONO
✅ [31] ARTICO ARTE E CONSTRUCAO LTDA
✅ [32] ART-VIVA
✅ [33] ATEMPO MOBILIARIO
✅ [34] ATTICA HOUSE
✅ [35] AUSTHER COMPUTACAO GRAFICA LTDA
✅ [36] AXIS
✅ [37] AXIS
✅ [38] AXIS
✅ [39] AXIS
✅ [40] B F S T MOVEIS LTDA
✅ 